# Kuhn Poker: CFR e ISMCTS

## El juego

Kuhn Poker es un juego de **suma cero**, **2 jugadores** e **información imperfecta** (cada jugador sólo conoce su propia carta):

- **Baraja:** J (0), Q (1), K (2). Cada jugador recibe 1 carta.
- **Acciones:** `p` = pasar (0), `b` = apostar (1).
- **Turno:** agente_0 actúa primero; agente_1 segundo. Si agente_0 pasó y agente_1 apuesta, agente_0 actúa una vez más.
- **Observación:** carta propia + historial público. Ejemplo: agente_0 tiene Q, historial `pb` → observación `'1pb'`.
- **Recompensas:** `pp`/`bp` → ±1 por carta más alta; `pbp` → ±1 para agente_1; `pbb`/`bb` → ±2 por carta más alta.

## Equilibrio de Nash

El juego tiene un equilibrio analíticamente conocido. El valor del juego es **−1/18 ≈ −0.056** para agente_0 (desventaja de actuar primero). Las políticas óptimas (parámetro α = 1/3):

| Agente | Info set | Prob. de apostar/call |
|--------|----------|-----------------------|
| 0 | `0` J inicio | 1/3 |
| 0 | `1` Q inicio | 0 |
| 0 | `2` K inicio | 1 |
| 0 | `0pb` J tras apuesta | 0 |
| 0 | `1pb` Q tras apuesta | 1/9 |
| 0 | `2pb` K tras apuesta | 1 |
| 1 | `0p` J tras paso | 0 |
| 1 | `1p` Q tras paso | 1/3 |
| 1 | `2p` K tras paso | 1 |
| 1 | `0b` J tras apuesta | 0 |
| 1 | `1b` Q tras apuesta | 1/3 |
| 1 | `2b` K tras apuesta | 1 |

## Los algoritmos

- **CFR:** calcula el Nash eq iterativamente minimizando regret contrafactual. Se entrena offline y luego decide en O(1).
- **ISMCTS:** extiende MCTS a info imperfecta samplando cartas posibles del rival en cada simulación. Decide online.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullFormatter

from games.kuhn.kuhn import KuhnPoker
from agents.agent_random import RandomAgent
from agents.counterfactualregret import CounterFactualRegret
from agents.ismcts import InformationSetMCTS

game = KuhnPoker(initial_player=0)  # agent_0 siempre actúa primero
game.reset()

NASH_0 = -1 / 18  # valor teórico para el primer jugador
NASH_1 = +1 / 18

FIGURES_DIR = os.path.join('..', 'informe', 'figures', 'KuhnPoker')
os.makedirs(FIGURES_DIR, exist_ok=True)

def savefig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Guardada: {path}')

PLAIN = FuncFormatter(lambda v, _: f'{v:g}')

def log_xticks(ax, ticks):
    ax.set_xticks(ticks)
    ax.xaxis.set_major_formatter(PLAIN)
    ax.xaxis.set_minor_formatter(NullFormatter())

## Función auxiliar

`play_games` recibe un diccionario de agentes ya creados y juega `n` partidas, \
devolviendo el reward y el tiempo promedio de decisión por agente.

In [ ]:
def play_games(game, agents, n=500):
    rows = []
    for _ in range(n):
        game.reset()
        times = {aid: [] for aid in agents}
        while not game.terminated():
            aid = game.agent_selection
            t0 = time.perf_counter()
            a = agents[aid].action()
            times[aid].append(time.perf_counter() - t0)
            game.step(a)
        row = {}
        for aid in agents:
            row[f'reward_{aid}'] = game.reward(aid)
            row[f'ms_{aid}'] = np.mean(times[aid]) * 1000 if times[aid] else 0.0
        rows.append(row)
    return pd.DataFrame(rows)

## Demo: una partida con agentes aleatorios

In [ ]:
rnd = {a: RandomAgent(game=game, agent=a) for a in game.agents}

game.reset()
game.render()
while not game.terminated():
    a = rnd[game.agent_selection].action()
    print(f'  {game.agent_selection} → {game.action_move(a)}')
    game.step(a)
game.render()
print('Recompensas:', game.rewards)

## CFR: entrenamiento y comparación con Nash

Se entrenan dos agentes CFR (uno por rol) con 5000 iteraciones \
y se compara cada política aprendida con los valores teóricos de Nash.

In [ ]:
NASH_BET = {
    'agent_0': {'0': 1/3, '1': 0.0, '2': 1.0, '0pb': 0.0, '1pb': 1/9, '2pb': 1.0},
    'agent_1': {'0p': 0.0, '1p': 1/3, '2p': 1.0, '0b': 0.0, '1b': 1/3, '2b': 1.0},
}

cfr = {a: CounterFactualRegret(game=game, agent=a) for a in game.agents}
for a, agent in cfr.items():
    agent.train(5000)
    print(f'{a}: {len(agent.node_dict)} nodos aprendidos')

rows = []
for a in game.agents:
    for info_set, nash_val in NASH_BET[a].items():
        cfr_val = cfr[a].node_dict[info_set].policy()[1] if info_set in cfr[a].node_dict else float('nan')
        rows.append(dict(agente=a, info_set=info_set,
                         cfr=round(cfr_val, 4), nash=round(nash_val, 4),
                         error=round(abs(cfr_val - nash_val), 4)))
pd.DataFrame(rows)

In [ ]:
# Convergencia de la prob. de apostar en los 3 info sets iniciales de agent_0
checkpoints = [50, 100, 200, 500, 1000, 2000, 5000]
info_sets_0 = {'0': 'J', '1': 'Q', '2': 'K'}

cfr0_conv = CounterFactualRegret(game=game, agent='agent_0')
history = {k: [] for k in info_sets_0}
prev = 0
for ck in checkpoints:
    cfr0_conv.train(ck - prev)
    prev = ck
    for k in info_sets_0:
        val = cfr0_conv.node_dict[k].policy()[1] if k in cfr0_conv.node_dict else float('nan')
        history[k].append(val)

fig, ax = plt.subplots(figsize=(8, 4))
for k, card in info_sets_0.items():
    ax.plot(checkpoints, history[k], marker='o', label=f"CFR '{k}' ({card})")
    ax.axhline(NASH_BET['agent_0'][k], linestyle='--', alpha=0.5,
               label=f"Nash '{k}' = {NASH_BET['agent_0'][k]:.3f}")
ax.set_xscale('log')
log_xticks(ax, checkpoints)
ax.set_xlabel('Iteraciones')
ax.set_ylabel('Prob. de apostar')
ax.set_title('CFR (agent_0): convergencia a Nash en la primera decisión')
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
fig.tight_layout()
savefig(fig, 'cfr_convergencia.png')
plt.show()

## ISMCTS: demostración

En cada simulación, ISMCTS samplea una carta posible para el rival usando `random_change`, \
luego corre MCTS sobre ese estado determinizado.

In [ ]:
def kuhn_sample(g, agent):
    return g.random_change(agent)

ismcts = {
    a: InformationSetMCTS(game=game, agent=a, simulations=100, rollouts=10,
                          sample_from_infoset=kuhn_sample)
    for a in game.agents
}

game.reset()
game.render()
while not game.terminated():
    a = ismcts[game.agent_selection].action()
    print(f'  {game.agent_selection} (obs={game.observe(game.agent_selection)!r}) → {game.action_move(a)}')
    game.step(a)
game.render()
print('Recompensas:', game.rewards)

## Experimento 1 — CFR vs Random: efecto de las iteraciones de entrenamiento

CFR como agente_0 (primer jugador) contra Random, variando `train_iters`.

In [ ]:
iter_grid = [100, 500, 1000, 2000, 5000]
N = 500

rows = []
for iters in iter_grid:
    cfr0 = CounterFactualRegret(game=game, agent='agent_0')
    cfr0.train(iters)
    df = play_games(game, {'agent_0': cfr0, 'agent_1': RandomAgent(game=game, agent='agent_1')}, n=N)
    r = df['reward_agent_0']
    rows.append(dict(iters=iters, avg_reward=r.mean(), win_rate=(r > 0).mean()))

df_exp1 = pd.DataFrame(rows)
df_exp1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_exp1.iters, df_exp1.avg_reward, marker='o', label='reward promedio')
ax.plot(df_exp1.iters, df_exp1.win_rate, marker='s', linestyle='--', label='win rate')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xscale('log')
log_xticks(ax, iter_grid)
ax.set_xlabel('Iteraciones de entrenamiento')
ax.set_title('CFR (agent_0) vs Random — efecto de las iteraciones')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
savefig(fig, 'exp1_cfr_vs_random.png')
plt.show()

## Experimento 2 — ISMCTS vs Random: efecto del número de simulaciones

ISMCTS como agente_0 contra Random, variando `simulations`.

In [ ]:
sim_grid = [5, 10, 25, 50, 100, 200]

rows = []
for sims in sim_grid:
    ism0 = InformationSetMCTS(game=game, agent='agent_0', simulations=sims,
                               rollouts=10, sample_from_infoset=kuhn_sample)
    df = play_games(game, {'agent_0': ism0, 'agent_1': RandomAgent(game=game, agent='agent_1')}, n=N)
    r = df['reward_agent_0']
    rows.append(dict(sims=sims, avg_reward=r.mean(), win_rate=(r > 0).mean(),
                     avg_time_ms=df['ms_agent_0'].mean()))

df_exp2 = pd.DataFrame(rows)
df_exp2

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_exp2.sims, df_exp2.avg_reward, marker='o', label='reward promedio')
axes[0].plot(df_exp2.sims, df_exp2.win_rate, marker='s', linestyle='--', label='win rate')
axes[0].axhline(0, color='gray', linestyle=':', linewidth=0.8)
axes[0].set_xscale('log')
log_xticks(axes[0], sim_grid)
axes[0].set_xlabel('Simulaciones')
axes[0].set_title('ISMCTS (agent_0) vs Random')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df_exp2.sims, df_exp2.avg_time_ms, marker='o', color='tab:orange')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
log_xticks(axes[1], sim_grid)
axes[1].yaxis.set_major_formatter(PLAIN)
axes[1].set_xlabel('Simulaciones')
axes[1].set_ylabel('ms por decisión')
axes[1].set_title('Tiempo de decisión de ISMCTS')
axes[1].grid(alpha=0.3)

fig.tight_layout()
savefig(fig, 'exp2_ismcts_vs_random.png')
plt.show()

## Experimento 3 — CFR vs ISMCTS

**3a:** CFR fijo (1000 iters, agent_0) vs ISMCTS (agent_1) con `simulations` variable.  
**3b:** ISMCTS fijo (100 sims, agent_1) vs CFR (agent_0) con `train_iters` variable.

El reward se reporta desde la perspectiva de agente_0. \
La línea punteada es el valor Nash teórico para agente_0 (−1/18).

In [ ]:
# 3a: CFR(1000) fijo vs ISMCTS variable
cfr0_ref = CounterFactualRegret(game=game, agent='agent_0')
cfr0_ref.train(1000)

rows_3a = []
for sims in [5, 10, 25, 50, 100, 200]:
    ism1 = InformationSetMCTS(game=game, agent='agent_1', simulations=sims,
                               rollouts=10, sample_from_infoset=kuhn_sample)
    df = play_games(game, {'agent_0': cfr0_ref, 'agent_1': ism1}, n=N)
    r = df['reward_agent_0']
    rows_3a.append(dict(sims=sims, avg_reward_cfr=r.mean()))

df_3a = pd.DataFrame(rows_3a)

# 3b: ISMCTS(100) fijo vs CFR variable
ism1_ref = InformationSetMCTS(game=game, agent='agent_1', simulations=100,
                               rollouts=10, sample_from_infoset=kuhn_sample)

rows_3b = []
for iters in [100, 500, 1000, 2000, 5000]:
    cfr0 = CounterFactualRegret(game=game, agent='agent_0')
    cfr0.train(iters)
    df = play_games(game, {'agent_0': cfr0, 'agent_1': ism1_ref}, n=N)
    r = df['reward_agent_0']
    rows_3b.append(dict(iters=iters, avg_reward_cfr=r.mean()))

df_3b = pd.DataFrame(rows_3b)
print(df_3a.to_string(index=False))
print()
print(df_3b.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_3a.sims, df_3a.avg_reward_cfr, marker='o', color='tab:blue', label='CFR agent_0 (1000 iters)')
axes[0].plot(df_3a.sims, -df_3a.avg_reward_cfr, marker='s', color='tab:orange', linestyle='--', label='ISMCTS agent_1')
axes[0].axhline(NASH_0, color='red', linestyle=':', alpha=0.6, label=f'Nash agent_0 = {NASH_0:.3f}')
axes[0].axhline(0, color='gray', linestyle=':', linewidth=0.6)
axes[0].set_xscale('log')
log_xticks(axes[0], [5, 10, 25, 50, 100, 200])
axes[0].set_xlabel('Simulaciones ISMCTS (agent_1)')
axes[0].set_ylabel('Reward promedio')
axes[0].set_title('CFR(1000, agent_0) vs ISMCTS(agent_1)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(df_3b.iters, df_3b.avg_reward_cfr, marker='o', color='tab:blue', label='CFR agent_0')
axes[1].plot(df_3b.iters, -df_3b.avg_reward_cfr, marker='s', color='tab:orange', linestyle='--', label='ISMCTS agent_1 (100 sims)')
axes[1].axhline(NASH_0, color='red', linestyle=':', alpha=0.6, label=f'Nash agent_0 = {NASH_0:.3f}')
axes[1].axhline(0, color='gray', linestyle=':', linewidth=0.6)
axes[1].set_xscale('log')
log_xticks(axes[1], [100, 500, 1000, 2000, 5000])
axes[1].set_xlabel('Iteraciones CFR (agent_0)')
axes[1].set_title('CFR(agent_0) vs ISMCTS(100, agent_1)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()
savefig(fig, 'exp3_cfr_vs_ismcts.png')
plt.show()

## Experimento 4 — Auto-juego: convergencia al equilibrio de Nash

En auto-juego simétrico, si ambos agentes juegan óptimamente el reward promedio de agente_0 \
debería converger a **−1/18 ≈ −0.056**.

- **CFR vs CFR:** convergencia garantizada al Nash eq.
- **ISMCTS vs ISMCTS:** sin garantías teóricas en info imperfecta.

In [ ]:
N_SELF = 1000
rows_cfr, rows_ism = [], []

for iters in [100, 500, 1000, 2000, 5000]:
    c0 = CounterFactualRegret(game=game, agent='agent_0')
    c1 = CounterFactualRegret(game=game, agent='agent_1')
    c0.train(iters)
    c1.train(iters)
    r = play_games(game, {'agent_0': c0, 'agent_1': c1}, n=N_SELF)['reward_agent_0']
    rows_cfr.append(dict(iters=iters, avg_reward=r.mean()))

for sims in [5, 10, 25, 50, 100, 200]:
    i0 = InformationSetMCTS(game=game, agent='agent_0', simulations=sims,
                             rollouts=10, sample_from_infoset=kuhn_sample)
    i1 = InformationSetMCTS(game=game, agent='agent_1', simulations=sims,
                             rollouts=10, sample_from_infoset=kuhn_sample)
    r = play_games(game, {'agent_0': i0, 'agent_1': i1}, n=N_SELF)['reward_agent_0']
    rows_ism.append(dict(sims=sims, avg_reward=r.mean()))

df_cfr_self = pd.DataFrame(rows_cfr)
df_ism_self = pd.DataFrame(rows_ism)
print('CFR vs CFR:')
print(df_cfr_self.to_string(index=False))
print('\nISMCTS vs ISMCTS:')
print(df_ism_self.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_cfr_self.iters, df_cfr_self.avg_reward, marker='o')
axes[0].axhline(NASH_0, color='red', linestyle='--', label=f'Nash = {NASH_0:.4f}')
axes[0].axhline(0, color='gray', linestyle=':', linewidth=0.6)
axes[0].set_xscale('log')
log_xticks(axes[0], [100, 500, 1000, 2000, 5000])
axes[0].set_xlabel('Iteraciones (ambos agentes CFR)')
axes[0].set_ylabel('Reward promedio de agent_0')
axes[0].set_title('CFR vs CFR: auto-juego')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df_ism_self.sims, df_ism_self.avg_reward, marker='s', color='tab:orange')
axes[1].axhline(NASH_0, color='red', linestyle='--', label=f'Nash = {NASH_0:.4f}')
axes[1].axhline(0, color='gray', linestyle=':', linewidth=0.6)
axes[1].set_xscale('log')
log_xticks(axes[1], [5, 10, 25, 50, 100, 200])
axes[1].set_xlabel('Simulaciones (ambos agentes ISMCTS)')
axes[1].set_ylabel('Reward promedio de agent_0')
axes[1].set_title('ISMCTS vs ISMCTS: auto-juego')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
savefig(fig, 'exp4_selfplay.png')
plt.show()

## Conclusiones

### Convergencia al Nash equilibrium

- **CFR** converge al Nash eq con suficientes iteraciones: las políticas aprendidas se acercan a los valores teóricos y el reward en auto-juego converge a −1/18. La tabla `df_nash` (celda de entrenamiento) cuantifica el error por information set.
- **ISMCTS** no tiene garantías de convergencia al Nash eq en información imperfecta. El Exp. 4 muestra si empíricamente se estabiliza cerca del valor teórico.

### Performance contra Random

Ambos agentes superan a Random incluso con presupuesto mínimo (Exp. 1 y 2). CFR es efectivo con pocas iteraciones porque el espacio de information sets de Kuhn Poker es muy pequeño (solo 12 nodos).

### CFR vs ISMCTS (Exp. 3)

CFR (offline) tiene tiempo de decisión O(1); ISMCTS (online) paga el costo en cada jugada. En un juego tan pequeño como Kuhn Poker, CFR bien entrenado tiende a tener ventaja por su convergencia garantizada al equilibrio.

| Aspecto | CFR | ISMCTS |
|---------|-----|--------|
| Entrenamiento | Offline (antes de jugar) | No (decide durante la partida) |
| Tiempo de decisión | O(1) — tabla de políticas | O(simulaciones) |
| Garantía Nash eq | Sí | No en info. imperfecta |
| Info. imperfecta | Nativo | Requiere determinización |